The Following Code is from Claudio

In [36]:
S11 = SymmetricGroup(11)
g1 = S11('(1,2,3,4,5,6,7,8,9,10,11)')
g2 = S11('(3,7,11,8)(4,10,5,6)')
G = S11.subgroup([g1, g2])
classes = G.conjugacy_classes()
sizes = {
str(cl.an_element()) : cl.cardinality()
for cl in classes
}
class_representatives = list(sizes.keys())
allreps = []
char_trivial = {g: 1 for g in class_representatives}
allreps.append(char_trivial)
def inner_product(chi: dict, psi: dict):
  return sum(sizes[g] * chi[g].conjugate() * psi[g] for g in class_representatives) / G.order()

We fixed the following basis so it doesnt change everytimre we run the above block

In [37]:
sizes = {'()': 1,
'(1,10,5,8,6)(2,9,3,11,4)' : 1584,
'(1,4)(2,11)(3,6)(9,10)' : 165,
'(1,6,4,3)(2,10,11,9)' : 990,
'(1,2,3,9,4,11,6,10)(7,8)' : 990,
'(1,10,6,11,4,9,3,2)(7,8)' : 990,
'(1,2,5,6,4,11,7,8,3,10,9)' : 720,
'(1,9,10,3,8,7,11,4,6,5,2)' : 720,
'(2,9,10)(3,4,5)(7,11,8)' : 440,
'(1,6)(2,7,9,11,10,8)(3,5,4)' :1320}

conjugacy_classes = {} #Regenerate Conjucacy classes
for key in sizes.keys():
  conjugacy_classes[key] = []
  for g in G:
    conjugacy_classes[key].append(g*S11(key)*g.inverse())
  #print(conjugacy_classes[key])

class_representatives = list(sizes.keys())
allreps = []
char_trivial = {g: 1 for g in class_representatives}
allreps.append(char_trivial)
def inner_product(chi: dict, psi: dict):
  return sum(sizes[g] * chi[g].conjugate() * psi[g] for g in class_representatives) / G.order()

We can easily write out the standard rep by taking the number of fixed points of each representative and subtracting $1$

In [38]:
char_standard = {'()': 10,
'(1,10,5,8,6)(2,9,3,11,4)' : 0,
'(1,4)(2,11)(3,6)(9,10)' : 2,
'(1,6,4,3)(2,10,11,9)' : 2,
'(1,2,3,9,4,11,6,10)(7,8)' : 0,
'(1,10,6,11,4,9,3,2)(7,8)' : 0,
'(1,2,5,6,4,11,7,8,3,10,9)' : -1,
'(1,9,10,3,8,7,11,4,6,5,2)' : -1,
'(2,9,10)(3,4,5)(7,11,8)' : 1,
'(1,6)(2,7,9,11,10,8)(3,5,4)' :-1}

We check that this is both an irrep and distinct from the trivial one. 

In [39]:
print(inner_product(char_standard,char_trivial))
print(inner_product(char_standard,char_standard))

0
1


Now that we have a nontrivial representation, we can compute the symmetric and exterior squares, $\text{Sym}^2(V)$ and $\wedge^2(V)$ respectively. Where $$\chi_{\text{Sym}^2(V)}(g) =  \frac 1 2 (\chi_V(g)^2 +\chi_V(g^2))\\\chi_{\wedge^2(V)}(g) =  \frac 1 2 (\chi_V(g)^2 -\chi_V(g^2))$$ (Use an eigenvalue argument for this proof) 

To do this, we first write a method to identify the conjugacy class $cl(g^2)$.

In [40]:
def Find_class(conj_classes, g): #Method to find what conjugacy class a particular element is in
  for key in conj_classes.keys():
    for h in conj_classes[key]:
      if h == g:
        #print(f"{str(g)} is in conjucacy class {key}")
        return key
  print("Not in any conj class??")
  return -1

def findSymSquared(char): #Method that returns Sym^2(V)
  char_symsquared = {}
  for key in char.keys():
    s = S11(key)*S11(key)
    char_symsquared[key] = 1/2 * (char[key]**2 + char[Find_class(conjugacy_classes,s)])
  return char_symsquared

We can now write methods to find the representation corresponding to the Symmetric and Exterior squares. It will also be useful for our process later to introduce a method that removes irreps from reducible reps (accounting for multiplicity), which will help us orthogonalize new reps. We also have a method that checks orthogonality and normalization with all current irreps.

In [41]:
def findExtSquared(char): #Method that returns Ext^2(V)
  char_ext = {}
  for key in char.keys():
    s = S11(key)*S11(key)
    char_ext[key] = 1/2 * (char[key]**2 - char[Find_class(conjugacy_classes,s)])
  return char_ext

def CheckWithAllIrreps(char_new: dict,irreps: list[dict]): #Method that prints out <V,V> and <V,Vi> for all other reps Vi to check normalization and orthogonality
  print(inner_product(char_new,char_new), "Normalization")
  for irrep in irreps:
    print(inner_product(char_new, irrep), f"Irrep with (): {irrep['()']}")

def OrthogonalizeWithKnownIrreps(char_new,irreps): #Method that "orthogonalizes" the given rep by removing the multiplicities of each known irrep within it. 
  char_temp = char_new
  c=0
  for irrep in irreps:
    c = inner_product(char_new,irrep)
    if c != 0:
      print(f"Subtracting {c} time(s) the rep with Tr(()) = {irrep['()']}")
    for key in char_new.keys():
      char_temp[key] -= c*irrep[key]
  return char_temp

In [42]:
irreps = [char_trivial,char_standard]
char_A = OrthogonalizeWithKnownIrreps(findSymSquared(char_standard),irreps)
CheckWithAllIrreps(char_A,irreps)
print(findSymSquared(char_standard))
CheckWithAllIrreps(findSymSquared(char_standard),[char_trivial,char_standard])
print(findExtSquared(char_standard))
CheckWithAllIrreps(findExtSquared(char_standard),[char_trivial,char_standard])


Subtracting 1 time(s) the rep with Tr(()) = 1
Subtracting 1 time(s) the rep with Tr(()) = 10
1 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
{'()': 55, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': 7, '(1,6,4,3)(2,10,11,9)': 3, '(1,2,3,9,4,11,6,10)(7,8)': 1, '(1,10,6,11,4,9,3,2)(7,8)': 1, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0, '(2,9,10)(3,4,5)(7,11,8)': 1, '(1,6)(2,7,9,11,10,8)(3,5,4)': 1}
3 Normalization
1 Irrep with (): 1
1 Irrep with (): 10
{'()': 45, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': -3, '(1,6,4,3)(2,10,11,9)': 1, '(1,2,3,9,4,11,6,10)(7,8)': -1, '(1,10,6,11,4,9,3,2)(7,8)': -1, '(1,2,5,6,4,11,7,8,3,10,9)': 1, '(1,9,10,3,8,7,11,4,6,5,2)': 1, '(2,9,10)(3,4,5)(7,11,8)': 0, '(1,6)(2,7,9,11,10,8)(3,5,4)': 0}
1 Normalization
0 Irrep with (): 1
0 Irrep with (): 10


The above representation, $A = \text{Sym}^2(V)-V-U$ is irreducible. We can do the same process for the Ext square.

In [43]:
irreps = [char_trivial,char_standard,char_A]
char_B = OrthogonalizeWithKnownIrreps(findExtSquared(char_standard),irreps)
CheckWithAllIrreps(char_B,irreps)

1 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
0 Irrep with (): 44


So the rep $B = \wedge^2(V)$ is irreducible and distinct as well! 

In [44]:
irreps = [char_trivial,char_standard,char_A,char_B]

testchars = []
names = []
for i, rep in enumerate(irreps[1:]):
    testchars.append(findExtSquared(rep))
    names.append(f'Ext^2 of rep with ():{rep['()']}')
    testchars.append(findSymSquared(rep))
    names.append(f'Sym^2 of rep with ():{rep['()']}')

print("reps that we have used are in the list ", irreps)
for i in range(len(testchars)):
   testchars[i] = OrthogonalizeWithKnownIrreps(testchars[i],irreps)
   print("irrep: ", names[i])
   CheckWithAllIrreps(testchars[i],irreps)
   print('\n')

print(testchars[3])
print(testchars[-1])
print('\n\n\n')
test = {}
for key in testchars[-1].keys():
   test[key] = testchars[-1][key]-testchars[3][key]
print(test)
print(names[-1],names[3])
print(inner_product(test,test))



reps that we have used are in the list  [{'()': 1, '(1,10,5,8,6)(2,9,3,11,4)': 1, '(1,4)(2,11)(3,6)(9,10)': 1, '(1,6,4,3)(2,10,11,9)': 1, '(1,2,3,9,4,11,6,10)(7,8)': 1, '(1,10,6,11,4,9,3,2)(7,8)': 1, '(1,2,5,6,4,11,7,8,3,10,9)': 1, '(1,9,10,3,8,7,11,4,6,5,2)': 1, '(2,9,10)(3,4,5)(7,11,8)': 1, '(1,6)(2,7,9,11,10,8)(3,5,4)': 1}, {'()': 10, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': 2, '(1,6,4,3)(2,10,11,9)': 2, '(1,2,3,9,4,11,6,10)(7,8)': 0, '(1,10,6,11,4,9,3,2)(7,8)': 0, '(1,2,5,6,4,11,7,8,3,10,9)': -1, '(1,9,10,3,8,7,11,4,6,5,2)': -1, '(2,9,10)(3,4,5)(7,11,8)': 1, '(1,6)(2,7,9,11,10,8)(3,5,4)': -1}, {'()': 44, '(1,10,5,8,6)(2,9,3,11,4)': -1, '(1,4)(2,11)(3,6)(9,10)': 4, '(1,6,4,3)(2,10,11,9)': 0, '(1,2,3,9,4,11,6,10)(7,8)': 0, '(1,10,6,11,4,9,3,2)(7,8)': 0, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0, '(2,9,10)(3,4,5)(7,11,8)': -1, '(1,6)(2,7,9,11,10,8)(3,5,4)': 1}, {'()': 45, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': -3, '(1,6,4,3)(2,10,

In [45]:
irreps = [char_trivial,char_standard,char_A,char_B]
char_C ={}
char_SymA = findSymSquared(char_A)
print(char_SymA)
CheckWithAllIrreps(char_SymA,irreps)
char_SymB = findSymSquared(char_B)
print(char_SymB)
CheckWithAllIrreps(char_SymB,irreps)
for key in char_A.keys():
  char_C[key] = -char_SymA[key]+char_SymB[key] +char_standard[key]
print(char_C)
CheckWithAllIrreps(char_C,irreps)
char_C = OrthogonalizeWithKnownIrreps(char_C,irreps)
print(char_C)
CheckWithAllIrreps(char_C,irreps)

{'()': 990, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': 30, '(1,6,4,3)(2,10,11,9)': 2, '(1,2,3,9,4,11,6,10)(7,8)': 0, '(1,10,6,11,4,9,3,2)(7,8)': 0, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0, '(2,9,10)(3,4,5)(7,11,8)': 0, '(1,6)(2,7,9,11,10,8)(3,5,4)': 0}
143 Normalization
1 Irrep with (): 1
3 Irrep with (): 10
8 Irrep with (): 44
4 Irrep with (): 45
{'()': 1035, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': 27, '(1,6,4,3)(2,10,11,9)': -1, '(1,2,3,9,4,11,6,10)(7,8)': 1, '(1,10,6,11,4,9,3,2)(7,8)': 1, '(1,2,5,6,4,11,7,8,3,10,9)': 1, '(1,9,10,3,8,7,11,4,6,5,2)': 1, '(2,9,10)(3,4,5)(7,11,8)': 0, '(1,6)(2,7,9,11,10,8)(3,5,4)': 0}
151 Normalization
1 Irrep with (): 1
2 Irrep with (): 10
8 Irrep with (): 44
4 Irrep with (): 45
{'()': 55, '(1,10,5,8,6)(2,9,3,11,4)': 0, '(1,4)(2,11)(3,6)(9,10)': -1, '(1,6,4,3)(2,10,11,9)': -1, '(1,2,3,9,4,11,6,10)(7,8)': 1, '(1,10,6,11,4,9,3,2)(7,8)': 1, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0

In [46]:
'''
 irreps = [char_trivial,char_standard,char_A,char_B]
char_shit ={}
char_SymA = findSymSquared(char_A)
for key in char_A.keys():
  char_shit[key] = char_SymB[key] +char_standard[key]

#print(char_shit)

#CheckWithAllIrreps(char_shit,irreps)
#char_shit2 = OrthogonalizeWithKnownIrreps(char_shit,irreps)
#print(char_shit2)
#CheckWithAllIrreps(char_shit,irreps)

#char_subshit = OrthogonalizeWithKnownIrreps(char_SymA,irreps)

print('Norms')
print(inner_product(char_SymA,char_SymA))
print(inner_product(char_shit,char_shit))

print('Original Inner Product')
print(inner_product(char_SymA,char_shit))

#print(inner_product(char_subshit,char_subshit))
print()

char_SymAPenis = OrthogonalizeWithKnownIrreps(char_SymA,irreps)
char_shitPenis = OrthogonalizeWithKnownIrreps(char_shit, irreps)

print('New Norms')
print(inner_product(char_SymAPenis, char_SymAPenis))
print(inner_product(char_shitPenis, char_shitPenis))

print('New Inner Product')
print(inner_product(char_SymAPenis, char_shitPenis))
'''


"\n irreps = [char_trivial,char_standard,char_A,char_B]\nchar_shit ={}\nchar_SymA = findSymSquared(char_A)\nfor key in char_A.keys():\n  char_shit[key] = char_SymB[key] +char_standard[key]\n\n#print(char_shit)\n\n#CheckWithAllIrreps(char_shit,irreps)\n#char_shit2 = OrthogonalizeWithKnownIrreps(char_shit,irreps)\n#print(char_shit2)\n#CheckWithAllIrreps(char_shit,irreps)\n\n#char_subshit = OrthogonalizeWithKnownIrreps(char_SymA,irreps)\n\nprint('Norms')\nprint(inner_product(char_SymA,char_SymA))\nprint(inner_product(char_shit,char_shit))\n\nprint('Original Inner Product')\nprint(inner_product(char_SymA,char_shit))\n\n#print(inner_product(char_subshit,char_subshit))\nprint()\n\nchar_SymAPenis = OrthogonalizeWithKnownIrreps(char_SymA,irreps)\nchar_shitPenis = OrthogonalizeWithKnownIrreps(char_shit, irreps)\n\nprint('New Norms')\nprint(inner_product(char_SymAPenis, char_SymAPenis))\nprint(inner_product(char_shitPenis, char_shitPenis))\n\nprint('New Inner Product')\nprint(inner_product(char_

So we see that the representation $\text{Sym}^2(A) - \wedge^2(V) - V$ is distinct and irreducible! We repeat this process but with the External squares. Which turns out to be another distinct irrep!

In [47]:
irreps = [char_trivial,char_standard,char_A,char_B,char_C]
char_D ={}
char_ExtA = findExtSquared(char_A)
CheckWithAllIrreps(char_ExtA,irreps)
char_ExtB = findExtSquared(char_B)
CheckWithAllIrreps(char_ExtB,irreps)
for key in char_A.keys():
  char_D[key] = char_ExtA[key]-char_ExtB[key]+char_standard[key]+char_B[key]
print(char_D)
#char_D = OrthogonalizeWithKnownIrreps(char_D,irreps)
print(char_D)
CheckWithAllIrreps(char_D,irreps)

118 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
4 Irrep with (): 44
6 Irrep with (): 45
7 Irrep with (): 55
131 Normalization
0 Irrep with (): 1
1 Irrep with (): 10
4 Irrep with (): 44
7 Irrep with (): 45
7 Irrep with (): 55
{'()': 11, '(1,10,5,8,6)(2,9,3,11,4)': 1, '(1,4)(2,11)(3,6)(9,10)': 3, '(1,6,4,3)(2,10,11,9)': -1, '(1,2,3,9,4,11,6,10)(7,8)': -1, '(1,10,6,11,4,9,3,2)(7,8)': -1, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0, '(2,9,10)(3,4,5)(7,11,8)': 2, '(1,6)(2,7,9,11,10,8)(3,5,4)': 0}
{'()': 11, '(1,10,5,8,6)(2,9,3,11,4)': 1, '(1,4)(2,11)(3,6)(9,10)': 3, '(1,6,4,3)(2,10,11,9)': -1, '(1,2,3,9,4,11,6,10)(7,8)': -1, '(1,10,6,11,4,9,3,2)(7,8)': -1, '(1,2,5,6,4,11,7,8,3,10,9)': 0, '(1,9,10,3,8,7,11,4,6,5,2)': 0, '(2,9,10)(3,4,5)(7,11,8)': 2, '(1,6)(2,7,9,11,10,8)(3,5,4)': 0}
1 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
0 Irrep with (): 44
0 Irrep with (): 45
0 Irrep with (): 55


In [48]:
irreps = [char_trivial,char_standard,char_A,char_B, char_C]
char_shit2 ={}
char_ExtB = findExtSquared(char_A)
for key in char_A.keys():
  char_shit2[key] = char_ExtA[key] + char_standard[key] + char_B[key]

#print(char_shit)

#CheckWithAllIrreps(char_shit,irreps)
#char_shit2 = OrthogonalizeWithKnownIrreps(char_shit,irreps)
#print(char_shit2)
#CheckWithAllIrreps(char_shit,irreps)

#char_subshit = OrthogonalizeWithKnownIrreps(char_SymA,irreps)

print('Norms')
print(inner_product(char_ExtB,char_ExtB))
print(inner_product(char_shit2,char_shit2))

print('Original Inner Product')
print(inner_product(char_ExtB,char_shit2))

#print(inner_product(char_subshit,char_subshit))
print()

char_ExtBPenis = OrthogonalizeWithKnownIrreps(char_ExtB,irreps)
char_shit2Penis = OrthogonalizeWithKnownIrreps(char_shit2, irreps)

print('New Norms')
print(inner_product(char_ExtBPenis, char_ExtBPenis))
print(inner_product(char_shit2Penis, char_shit2Penis))

print('New Inner Product')
print(inner_product(char_ExtBPenis, char_shit2Penis))

Norms
118
132
Original Inner Product
124

Subtracting 4 time(s) the rep with Tr(()) = 44
Subtracting 6 time(s) the rep with Tr(()) = 45
Subtracting 7 time(s) the rep with Tr(()) = 55
Subtracting 1 time(s) the rep with Tr(()) = 10
Subtracting 4 time(s) the rep with Tr(()) = 44
Subtracting 7 time(s) the rep with Tr(()) = 45
Subtracting 7 time(s) the rep with Tr(()) = 55
New Norms
17
17
New Inner Product
17


In [49]:
print(inner_product(char_ExtA,char_ExtA))
print(inner_product(char_ExtB,char_ExtB))

118
17


In [50]:
pip install pandas


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [51]:
import pandas as pd

irreps = [char_trivial,char_standard,char_A,char_B,char_C,char_D]
df = pd.DataFrame(irreps)
df



,(),"(1,10,5,8,6)(2,9,3,11,4)","(1,4)(2,11)(3,6)(9,10)","(1,6,4,3)(2,10,11,9)","(1,2,3,9,4,11,6,10)(7,8)","(1,10,6,11,4,9,3,2)(7,8)","(1,2,5,6,4,11,7,8,3,10,9)","(1,9,10,3,8,7,11,4,6,5,2)","(2,9,10)(3,4,5)(7,11,8)","(1,6)(2,7,9,11,10,8)(3,5,4)"
0,1,1,1,1,1,1,1,1,1,1
1,10,0,2,2,0,0,-1,-1,1,-1
2,44,-1,4,0,0,0,0,0,-1,1
3,45,0,-3,1,-1,-1,1,1,0,0
4,55,0,-1,-1,1,1,0,0,1,-1
5,11,1,3,-1,-1,-1,0,0,2,0


In [52]:
char_cubed = {}
for key in char_A.keys():
  char_cubed[key] = char_standard[key]**3
char_cubed = OrthogonalizeWithKnownIrreps(char_cubed,irreps)
print(char_cubed)
CheckWithAllIrreps(char_cubed,irreps)
g = S11('(1,2,3,9,4,11,6,10)(7,8)')
current = g
for i in range(9):
  print(current,"  in ",Find_class(conjugacy_classes,current))
  current = current*g
h = S11('(1,10,6,11,4,9,3,2)(7,8)')
current2 = h
for i in range(9):
  print(current2,"  in ",Find_class(conjugacy_classes,current2))
  current2 = current2*h

Subtracting 1 time(s) the rep with Tr(()) = 1
Subtracting 4 time(s) the rep with Tr(()) = 10
Subtracting 6 time(s) the rep with Tr(()) = 44
Subtracting 6 time(s) the rep with Tr(()) = 45
Subtracting 6 time(s) the rep with Tr(()) = 55
Subtracting 1 time(s) the rep with Tr(()) = 11
{'()': 84, '(1,10,5,8,6)(2,9,3,11,4)': 4, '(1,4)(2,11)(3,6)(9,10)': -4, '(1,6,4,3)(2,10,11,9)': 0, '(1,2,3,9,4,11,6,10)(7,8)': 0, '(1,10,6,11,4,9,3,2)(7,8)': 0, '(1,2,5,6,4,11,7,8,3,10,9)': -4, '(1,9,10,3,8,7,11,4,6,5,2)': -4, '(2,9,10)(3,4,5)(7,11,8)': -6, '(1,6)(2,7,9,11,10,8)(3,5,4)': 2}
10 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
0 Irrep with (): 44
0 Irrep with (): 45
0 Irrep with (): 55
0 Irrep with (): 11
(1,2,3,9,4,11,6,10)(7,8)   in  (1,2,3,9,4,11,6,10)(7,8)
(1,3,4,6)(2,9,11,10)   in  (1,6,4,3)(2,10,11,9)
(1,9,6,2,4,10,3,11)(7,8)   in  (1,2,3,9,4,11,6,10)(7,8)
(1,4)(2,11)(3,6)(9,10)   in  (1,4)(2,11)(3,6)(9,10)
(1,11,3,10,4,2,6,9)(7,8)   in  (1,10,6,11,4,9,3,2)(7,8)
(1,6,4,3)(2,10,11,9)   

In [53]:
char_quad = {}
for key in char_A.keys():
  char_quad[key] = char_standard[key]**5
char_cubed = OrthogonalizeWithKnownIrreps(char_quad,irreps)
print(char_quad)
CheckWithAllIrreps(char_quad,irreps)

Subtracting 17 time(s) the rep with Tr(()) = 1
Subtracting 136 time(s) the rep with Tr(()) = 10
Subtracting 558 time(s) the rep with Tr(()) = 44
Subtracting 570 time(s) the rep with Tr(()) = 45
Subtracting 690 time(s) the rep with Tr(()) = 55
Subtracting 137 time(s) the rep with Tr(()) = 11
{'()': 8964, '(1,10,5,8,6)(2,9,3,11,4)': 404, '(1,4)(2,11)(3,6)(9,10)': -500, '(1,6,4,3)(2,10,11,9)': 0, '(1,2,3,9,4,11,6,10)(7,8)': 0, '(1,10,6,11,4,9,3,2)(7,8)': 0, '(1,2,5,6,4,11,7,8,3,10,9)': -452, '(1,9,10,3,8,7,11,4,6,5,2)': -452, '(2,9,10)(3,4,5)(7,11,8)': -558, '(1,6)(2,7,9,11,10,8)(3,5,4)': 250}
112858 Normalization
0 Irrep with (): 1
0 Irrep with (): 10
0 Irrep with (): 44
0 Irrep with (): 45
0 Irrep with (): 55
0 Irrep with (): 11


In [54]:

def OrthogonalizeWithKnownIrreps(char_new,irreps):
  char_temp = char_new
  c=0
  for irrep in irreps:
    c = inner_product(char_new,irrep)
    # if c != 0:
    #   print(f"Subtracting {c} time(s) the rep with Tr(()) = {irrep['()']}")
    for key in char_new.keys():
      char_temp[key] -= c*irrep[key]
  return char_temp

testlist = []
goodlist=[]
import numpy as np
for i in range(10):
  print(i)
  for irrep in irreps:
    char_temp = irrep
    for j in range(i*2+1):
      r = np.random.randint(1)
      if r>= 0.5:
        char_temp = findSymSquared(char_temp)
      else:
        char_temp = findExtSquared(char_temp)
    char_temp = OrthogonalizeWithKnownIrreps(char_temp,irreps)
    testlist.append(char_temp)

print(len(testlist))
for char in testlist:
  c = inner_product(char,char)
  if 0 < c <= 17:
    print("WOAHHH",c)
    goodlist.append(char)


df2 = pd.DataFrame(goodlist)
df2
  

0
1
2
3
4
5
6
7
8
9
60
WOAHHH 16
WOAHHH 16


,(),"(1,10,5,8,6)(2,9,3,11,4)","(1,4)(2,11)(3,6)(9,10)","(1,6,4,3)(2,10,11,9)","(1,2,3,9,4,11,6,10)(7,8)","(1,10,6,11,4,9,3,2)(7,8)","(1,2,5,6,4,11,7,8,3,10,9)","(1,9,10,3,8,7,11,4,6,5,2)","(2,9,10)(3,4,5)(7,11,8)","(1,6)(2,7,9,11,10,8)(3,5,4)"
0,104,4,-8,0,0,0,-6,-6,-4,4
1,104,4,-8,0,0,0,-6,-6,-4,4
